# Fair IU-Xray Archive Benchmark (Kaggle GPU)

Train M2Transformer, R2Gen and Pure Transformer sequentially on the same mounted 2,351/293/293 split. Select **GPU T4** in Kaggle notebook settings and attach a dataset containing `processed_train.json`, `processed_val.json`, `processed_test.json`, and the referenced `images/` files. The archive vocabulary is read from the repository and is never modified.

The selected protocol uses seed 42, batch size 16, learning rate `1e-4`, up to 18 epochs and patience 5 by validation CIDEr. Pure Transformer retains its multi-criteria objective; the two baseline architectures use report-generation cross-entropy.

In [ ]:
!pip -q install pycocoevalcap matplotlib

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Shrio182/DACN.git"
REPO_DIR = Path("/kaggle/working/DACN")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
%cd /kaggle/working/DACN

In [ ]:
import glob
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator (recommended: NVIDIA T4) in notebook settings.")
gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)
if "T4" not in gpu_name:
    print("Notice: this runtime is not a T4; measured timing will differ from the requested setup.")
if shutil.which("java") is None:
    raise RuntimeError("Java is required by the METEOR evaluator; enable a Kaggle image with Java installed.")

matches = glob.glob("/kaggle/input/**/processed_train.json", recursive=True)
if len(matches) == 1:
    DATA_DIR = str(Path(matches[0]).parent)
else:
    DATA_DIR = "/kaggle/input/iu-xray-archive-processed"  # Set this to the attached dataset directory.
print("Dataset directory:", DATA_DIR)
required = ["processed_train.json", "processed_val.json", "processed_test.json"]
missing = [name for name in required if not (Path(DATA_DIR) / name).is_file()]
if missing:
    raise FileNotFoundError(f"Set DATA_DIR to the mounted IU-Xray dataset; missing: {missing}")

In [ ]:
!python run_fair_benchmark.py --data-dir "{DATA_DIR}" --vocab artifacts/archive/vocab_archive.json --models m2transformer r2gen pure_transformer --epochs 18 --batch-size 16 --patience 5

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

csv_path = Path("artifacts/fair_benchmark_archive.csv")
png_path = Path("artifacts/fair_benchmark_archive.png")
for path in (csv_path, png_path):
    if not path.is_file():
        raise FileNotFoundError(f"Benchmark output was not created: {path}")
zip_path = Path("/kaggle/working/fair_benchmark_archive.zip")
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(csv_path, arcname=csv_path.name)
    archive.write(png_path, arcname=png_path.name)
print("Download:", zip_path)